# Quantization

In [12]:
import numpy as np
import glob
from tqdm import tqdm
from PIL import Image

mapping = {}
for i in range(1000):
    i_str = f'{i:03}'
    mapping[i_str] = i
mapping["010"] = 20
mapping["101"] = 102
    
def pixel_to_char(pixel):
    pixel = np.array(pixel) // 26
    pixel = "".join([str(x) for x in pixel])
    pixel_value = mapping[pixel]
    pixel_char = chr(pixel_value)
    return pixel_char

paths = glob.glob("./data/*.jpg")
SQUEEZE = 2
im_array = np.array(Image.open(paths[2]))[::SQUEEZE,::SQUEEZE]

with open(f"test{SQUEEZE}.txt","a") as f:
    for i in tqdm(range(len(paths))):
        im_array = np.array(Image.open(paths[i]))[::SQUEEZE,::SQUEEZE]
        im_list = list(im_array)
        for line in im_list:
            line = [pixel_to_char(x) for x in line]
            f.write("".join(line))
            f.write("\n")
        f.write("e")
        # break

100%|██████████| 100/100 [00:03<00:00, 25.42it/s]


# Tokenizer Training

In [13]:
from tokenizers import ByteLevelBPETokenizer

SQUEEZE = 2
VOCAB_SIZE = 30000
files = [f"test{SQUEEZE}.txt"]
# Initialize a tokenizer
tokenizer = ByteLevelBPETokenizer()
# Customize training
tokenizer.train(files=files, vocab_size=VOCAB_SIZE)
# Save files to disk
tokenizer.save_model("./", f"tokenizer-faces_color-{SQUEEZE}-{VOCAB_SIZE}")

['./tokenizer-faces_color-2-30000-vocab.json',
 './tokenizer-faces_color-2-30000-merges.txt']

# GPT training

In [4]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Model, GPT2Config

SQUEEZE = 2
VOCAB_SIZE = 30000
tokenizer = GPT2Tokenizer(vocab_file = f"tokenizer-faces_color-{SQUEEZE}-{VOCAB_SIZE}-vocab.json",
                          merges_file = f"tokenizer-faces_color-{SQUEEZE}-{VOCAB_SIZE}-merges.txt"
                         )

eos_id = tokenizer.encode("e")[0]

N_TOKENS = 1024 * 5

configuration = GPT2Config(vocab_size = VOCAB_SIZE,
                           n_positions = N_TOKENS,
                           n_embd = 1024,
                           n_layer = 8,
                           n_head = 4,
                           n_inner = None,
                           activation_function = 'gelu_new',
                           resid_pdrop = 0., #0.1,
                           embd_pdrop = 0.,
                           attn_pdrop = 0.,
                           layer_norm_epsilon = 1e-05,
                           initializer_range = 0.02,
                           summary_type = 'cls_index',
                           summary_use_proj = True,
                           summary_activation = None,
                           summary_proj_to_labels = True,
                           summary_first_dropout = 0.,
                           scale_attn_weights = True,
                           use_cache = True,
                           bos_token_id = eos_id,
                           eos_token_id = eos_id,
                           scale_attn_by_inverse_layer_idx = True,
                           reorder_and_upcast_attn = False)

# Initializing a model from the configuration
model = GPT2LMHeadModel(configuration)

In [5]:
from transformers import TextDataset, DataCollatorForLanguageModeling

args = {
    "epochs": 500,
    "bs": 1 * 2,
    # "warmup": 100,
    "grad_accum": 128,
    "tokens": N_TOKENS,
    "training": True,
    "lr": 1e-4,
    "wd": 1e-3,
    
}

# Сохраним обучающие данные в .txt файл 
train_path = f"test{SQUEEZE}.txt"
# train_path = f'data/faces_short.txt'

# Создание датасета
train_dataset = TextDataset(
    tokenizer=tokenizer, 
    file_path=train_path, 
    block_size=args["tokens"]
)

# Создание даталодера (нарезает текст на оптимальные по длине куски)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

model.training = args["training"]

/Users/yubo/miniconda3/envs/ml/lib/python3.8/site-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [11]:
import torch
from transformers import Trainer, TrainingArguments


# model = torch.nn.DataParallel(model, device_ids=[0, 1, 2, 3,  4, 5, 6, 7])

training_args = TrainingArguments(
    do_train=True,
    do_eval=False,
    output_dir="./finetuned", #The output directory
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=args["epochs"], # number of training epochs
    per_device_train_batch_size=args["bs"], # batch size for training
    # per_device_eval_batch_size=1,  # batch size for evaluation
    # warmup_steps=args["warmup"],# number of warmup steps for learning rate scheduler
    # gradient_accumulation_steps=args["grad_accum"], # to make "virtual" batch size larger
    # label_smoothing_factor=0.1,
    # deepspeed=True,
    dataloader_num_workers=20,
    fp16 = False,
    logging_steps = 50,
    learning_rate = args["lr"],
    weight_decay = args["wd"],
    save_steps = 5000,
    # label_smoothing_factor = 1e-2
    )

# from accelerate import DataLoaderConfiguration
# from accelerate import Accelerator
# dataloader_config = DataLoaderConfiguration(dispatch_batches=False)
# accelerator = Accelerator(..., dataloader_config=dataloader_config)


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    # is_model_parallel = True
    # optimizers = (torch.optim.AdamW(model.parameters(),lr=args["lr"]),None) # Optimizer and lr scheduler
)

TypeError: __init__() got an unexpected keyword argument 'dispatch_batches'